# SPX Implied Volatility Surface — Walkthrough

Builds an **arbitrage-free implied-volatility surface** from a real SPX option chain, stage by stage:

`raw chain → clean → forward (put-call parity) → IV solve → no-arb checks → SVI fit → surface + risk-neutral density`

SPX options are European and cash-settled, so Black–Scholes is the correct pricer. Run the cell below after fetching at least one snapshot (`python -m volsurface.pipeline`).

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from volsurface.config import Config
from volsurface.fetcher import load_raw, fetch_chain
from volsurface.cleaner import clean_chain
from volsurface.forward import extract_and_attach
from volsurface.iv_solver import select_otm, solve_iv_frame
from volsurface.arbitrage import check_arbitrage, drop_butterfly_violations
from volsurface.svi import fit_surface, params_from_row
from volsurface import surface, density

cfg = Config.load()

## 1. Ingest
Load the most recent saved snapshot (or call `fetch_chain(cfg)` to pull a fresh one).

In [ ]:
files = sorted(glob.glob(str(cfg.paths.resolve('raw_dir') / 'spx_*.parquet')))
chain = load_raw(files[-1]) if files else fetch_chain(cfg)
print('as-of', chain.asof, '| spot', round(chain.spot, 2), '| raw quotes', chain.n_quotes,
      '| expiries', len(chain.expiries))
chain.quotes.head()

## 2. Clean & filter
Drop one-sided/crossed/illiquid quotes, compute mid prices, restrict moneyness. The rejection report attributes every dropped quote to a rule.

In [ ]:
cleaned, rejection = clean_chain(chain, cfg)
print(rejection)

## 3. Forward & discount from put-call parity
Regress (C - P) on K per expiry: slope = -e^{-rT}, intercept = e^{-rT}·F. The market sets the forward (captures dividends/carry) — we never assume F = S·e^{rT}.

In [ ]:
fwd = extract_and_attach(chain, cleaned, cfg)
fwd.forwards[['T','F','disc','rate','n_pairs','method']].head(10).round(4)

In [ ]:
f = fwd.forwards[fwd.forwards.method == 'parity']
plt.figure(figsize=(7,4))
plt.plot(f['T'], f['F'], 'o-', label='market-implied forward F(T)')
plt.axhline(chain.spot, color='gray', ls='--', label='spot')
plt.xlabel('T (yrs)'); plt.ylabel('forward'); plt.legend(); plt.title('Forward curve'); plt.show()

## 4. Implied vol (OTM legs)
Pick the OTM leg per strike and invert Black–Scholes for IV; add total variance w = iv²·T.

In [ ]:
iv_all = solve_iv_frame(select_otm(fwd.quotes), cfg)
print('OTM IV points:', len(iv_all), 'across', iv_all['expiry'].nunique(), 'expiries')

## 5. No-arbitrage diagnostics (raw grid)
Count calendar (total variance non-decreasing in T) and butterfly (call price convex in K) violations on the *raw* inverted grid — the baseline the SVI fit will remove.

In [ ]:
flagged, arb = check_arbitrage(iv_all, cfg)
print(arb)
iv_clean = drop_butterfly_violations(flagged)

## 6. SVI calibration
Fit Gatheral's raw SVI per expiry slice; certify each is butterfly-free via g(k).

In [ ]:
svi = fit_surface(iv_clean, cfg)
print('slices fit:', len(svi), '| all butterfly-free:', bool(svi.butterfly_free.all()),
      '| median RMSE (vol pts):', round(float(svi.rmse_vol.median())*100, 2))
svi[['T','a','b','rho','m','sigma','n_points','rmse_vol','butterfly_free']].head(8).round(4)

## 7. Surface, smile & ATM term structure

In [ ]:
expiry = sorted(svi['expiry'].unique())[len(svi)//3]
row = svi[svi.expiry == expiry].iloc[0]
p = params_from_row(row)
pts = surface.compute_iv_band(iv_clean[iv_clean.expiry == expiry].sort_values('log_moneyness'), cfg)
k = pts['log_moneyness'].to_numpy()
kk = np.linspace(k.min(), k.max(), 200)

plt.figure(figsize=(8,5))
ok = pts['iv_bid'].notna() & pts['iv_ask'].notna()
plt.fill_between(k[ok], pts['iv_bid'][ok]*100, pts['iv_ask'][ok]*100, color='gray', alpha=0.3, label='bid/ask band')
plt.scatter(k, pts['iv']*100, s=14, label='market mid IV')
plt.plot(kk, p.implied_vol(kk, row['T'])*100, 'r', lw=2, label='SVI fit')
plt.xlabel('k = log(K/F)'); plt.ylabel('IV (%)'); plt.legend()
plt.title(f'Smile {expiry}  T={row["T"]:.3f}y'); plt.show()

In [ ]:
ts = surface.atm_term_structure(svi)
plt.figure(figsize=(7,4))
plt.plot(ts['T'], ts['atm_iv']*100, 'o-')
plt.xlabel('T (yrs)'); plt.ylabel('ATM IV (%)'); plt.title('ATM term structure'); plt.grid(alpha=0.3); plt.show()

## 8. Risk-neutral density (Breeden–Litzenberger via SVI)
The market-implied distribution of S_T, computed in closed form from each SVI slice.

In [ ]:
kk = np.linspace(-0.6, 0.4, 400)
plt.figure(figsize=(8,5))
for i in sorted({0, len(svi)//2, len(svi)-1}):
    row = svi.sort_values('T').iloc[i]; p = params_from_row(row)
    K, fK = density.rnd_strike(p, row['F'], kk)
    plt.plot(K, fK, label=f"{row['expiry']} (T={row['T']:.2f}y)")
plt.xlabel('SPX level $S_T$'); plt.ylabel('risk-neutral density'); plt.legend()
plt.title('Market-implied risk-neutral density'); plt.grid(alpha=0.3); plt.show()